# Explicit differentiation of transient elastodynamics

In this tutorial, we reconstruct the axial distribution of Young's modulus in a three-dimensional elastic waveguide from transient displacement observations. The displacement trajectory is computed using a lumped-mass finite element discretization and the explicit central-difference method. A uniform material is first used to verify the forward solution against the first longitudinal mode. The heterogeneous inverse problem assigns one modulus parameter to each axial element layer. Gradients of the observation loss are obtained by reverse-mode automatic differentiation (AD) through the time integration steps, while gradient checkpointing is used to trade recomputation for reduced memory. The implementation uses [JAX](https://github.com/jax-ml/jax) and [JAX-FEM](https://github.com/deepmodeling/jax-fem).

## Problem formulations

### Forward problem

Consider the free vibration of a three-dimensional elastic waveguide $\Omega=[0,L_x]\times[0,L_y]\times[0,L_z]$. The small-strain elastodynamic equations are

$$
\rho\ddot{\boldsymbol u}=\nabla\cdot\boldsymbol\sigma,
\qquad
\boldsymbol\sigma=\lambda\operatorname{tr}(\boldsymbol\varepsilon)\boldsymbol I+2\mu\boldsymbol\varepsilon,
\qquad
\boldsymbol\varepsilon=\frac{1}{2}\left(\nabla\boldsymbol u+\nabla\boldsymbol u^\top\right),
$$

where $\rho$ is the mass density, $\boldsymbol I$ is the identity tensor, and the Lamé parameters $\lambda$ and $\mu$ are determined by the Young's modulus $E$ and the Poisson's ratio $\nu$. The left end satisfies $u_x=0$, while the normal transverse displacement is constrained on the lateral faces. The right end is traction-free. The waveguide is released from the initial state

$$
u_x(\boldsymbol x,0)=A\sin(kx),
\qquad
\dot{\boldsymbol u}(\boldsymbol x,0)=\boldsymbol 0,
\qquad
k=\frac{\pi}{2L_x}.
$$

The waveguide is released from rest, and the initial elastic strain energy is periodically exchanged with kinetic energy during the resulting undamped free vibration.

For a uniform modulus $E$, the first fixed-free longitudinal mode has the exact solution

$$
u_x(\boldsymbol x,t)=A\sin(kx)\cos(c_p k t),
\qquad
c_p=\sqrt{\frac{E(1-\nu)}{(1+\nu)(1-2\nu)\rho}}.
$$

Let $\delta\boldsymbol u$ be an admissible virtual displacement and $\delta\boldsymbol\varepsilon=\frac{1}{2}(\nabla\delta\boldsymbol u+\nabla\delta\boldsymbol u^\top)$ its virtual strain. The weak form is

$$
\int_\Omega \rho\,\delta\boldsymbol u\cdot\ddot{\boldsymbol u}\,\mathrm d\Omega
+\int_\Omega \delta\boldsymbol\varepsilon:\boldsymbol\sigma\,\mathrm d\Omega=0.
$$

Using the finite element approximation $\boldsymbol u^h=\sum_a N_a\boldsymbol U_a$, where $N_a$ is the nodal shape function and $\boldsymbol U_a$ is the nodal displacement vector, gives the semi-discrete system

$$
\boldsymbol M\ddot{\boldsymbol U}+\boldsymbol F_{\mathrm{int}}=\boldsymbol 0,
$$

where the scalar mass coefficient, the corresponding matrix block, and the nodal internal force are

$$
m_{ab}=\int_\Omega \rho N_aN_b\,\mathrm d\Omega,\qquad
\boldsymbol M_{ab}=m_{ab}\boldsymbol I,\qquad
\boldsymbol F_{\mathrm{int},a}=\int_\Omega \boldsymbol B_a^\top\boldsymbol\sigma\,\mathrm d\Omega.
$$

Here $\boldsymbol B_a$ is the strain-displacement operator associated with node $a$. Row-sum lumping replaces the consistent mass matrix by the diagonal matrix

$$
\boldsymbol M_L=\operatorname{diag}(m_1\boldsymbol I,\ldots,m_{N_n}\boldsymbol I),\qquad
m_a=\sum_bm_{ab}=\int_\Omega \rho N_a\,\mathrm d\Omega.
$$

Here $N_n$ is the total number of finite element nodes. The acceleration at time step $n$ is therefore obtained without solving a linear system,

$$
\boldsymbol A^n=-\boldsymbol M_L^{-1}\boldsymbol F_{\mathrm{int}}^n.
$$

For a time-step size $\Delta t$, the fictitious previous state is initialized to second-order accuracy,

$$
\boldsymbol U^{-1}=\boldsymbol U^0-\Delta t\dot{\boldsymbol U}^0+\frac{1}{2}\Delta t^2\boldsymbol A^0.
$$

The central-difference displacement update is then

$$
\boldsymbol U^{n+1}=2\boldsymbol U^n-\boldsymbol U^{n-1}-\Delta t^2\boldsymbol M_L^{-1}\boldsymbol F_{\mathrm{int}}^n.
$$

### Inverse problem

The waveguide is divided into $N_x$ axial element layers, with one Young's modulus assigned to each layer, $\boldsymbol E=(E_1,\ldots,E_{N_x})$. Let $\widehat{u}_r^n$ denote the observed axial displacement at probe $r$ and time step $n$. The modulus distribution is identified from

$$
\boldsymbol E^*=\arg\min_{\boldsymbol E}\mathcal L(\boldsymbol E),\qquad
\mathcal L(\boldsymbol E)=\frac{1}{N_o(N+1)}\sum_{r=1}^{N_o}\sum_{n=0}^{N}\left[\frac{u_r^n(\boldsymbol E)-\widehat{u}_r^n}{u_{\mathrm{rms}}}\right]^2.
$$

Here $N_o$ is the number of observation points, $N$ is the number of time steps, $u_r^n$ is the predicted axial displacement at probe $r$, $u_{\mathrm{rms}}$ is the root-mean-square observed displacement, and $E_0$ is the reference modulus. The target modulus varies linearly along the waveguide,

$$
E^*(x)=E_0\left(1-0.4\frac{x}{L_x}\right).
$$

The axial displacement is observed at the center of every free cross-section,

$$
\boldsymbol x_r=\left(\frac{rL_x}{N_x},\frac{L_y}{2},\frac{L_z}{2}\right),\qquad r=1,\ldots,N_x,
$$

so that $N_o=N_x$.

### Explicit differentiation and checkpointing

The central-difference scheme defines the explicit recurrence $\boldsymbol S^{n+1}=\mathcal G(\boldsymbol S^n,\boldsymbol E)$, where $\boldsymbol S^n=(\boldsymbol U^n,\boldsymbol U^{n-1})$ contains the states required by the next time step. Reverse-mode AD computes $\nabla_{\boldsymbol E}\mathcal L$ by backpropagating through this recurrence. Without checkpointing, the reverse pass retains intermediate values from the complete trajectory. Wrapping a block of time steps with `jax.checkpoint` allows its internal values to be discarded and recomputed from the block input during backpropagation. Shorter blocks reduce the temporary storage required within each block but increase the number of block transitions.

## Implementation

We first import some useful packages

In [ ]:
# Standard library
import logging
import os
import time

# Scientific computing
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

# JAX
import jax
import jax.numpy as jnp
import optax

# JAX-FEM
from jax_fem import logger
from jax_fem.generate_mesh import Mesh, box_mesh, get_meshio_cell_type
from jax_fem.problem import Problem
logger.setLevel(logging.WARNING)

jax.config.update("jax_enable_x64", True)

print(f"JAX backend: {jax.default_backend()}")

### Mesh

The waveguide has length $L_x=1$ and a square cross-section of width $0.1$. A structured $80\times4\times4$ grid is generated using `jax_fem.generate_mesh.box_mesh` and stored in a `jax_fem.generate_mesh.Mesh` object with trilinear hexahedral elements. The dimensions and material properties use a consistent nondimensional system.

In [ ]:
# Geometry
Lx, Ly, Lz = 1.0, 0.1, 0.1
Nx, Ny, Nz = 80, 4, 4

# Structured hexahedral mesh
ele_type = "HEX8"
cell_type = get_meshio_cell_type(ele_type)
meshio_mesh = box_mesh(Nx, Ny, Nz, Lx, Ly, Lz)
points = meshio_mesh.points
cells = meshio_mesh.cells_dict[cell_type]
mesh = Mesh(points, cells)

# Material properties
E0, nu, rho = 1.0, 0.3, 1.0

print(f"Elements: {mesh.cells.shape[0]}")
print(f"Nodes: {mesh.points.shape[0]}")

In [ ]:
# Visualize the finite element mesh
plotter = pv.Plotter(notebook=True, window_size=(800, 400))
pv_mesh = pv.UnstructuredGrid(
    {getattr(pv.CellType, cell_type.upper()): cells}, points
)
plotter.add_mesh(
    pv_mesh, color="lightgray", show_edges=True, edge_color="gray", line_width=0.5
)
span = np.ptp(points, axis=0).max()
origin = points.min(0) - np.array([0.05 * span, 0.05 * span, 0.05 * span])
length = 0.08 * span
for direction, color, label in zip(np.eye(3), ("red", "green", "blue"), ("X", "Y", "Z")):
    plotter.add_mesh(pv.Arrow(start=origin, direction=direction, scale=length), color=color)
    plotter.add_mesh(
        pv.Text3D(label, height=0.025 * span, depth=0.005 * span,
                  center=origin + 1.25 * length * direction, normal=(0, -1, 0)),
        color="black",
    )
plotter.view_isometric(negative=True)
plotter.show(jupyter_backend="html")

### Boundary conditions

The axial displacement is fixed at $x=0$. The normal displacement is fixed on each lateral pair of faces, while the end $x=L_x$ remains traction-free. In JAX-FEM, `dirichlet_bc_info` groups the boundary location functions, constrained displacement components, and prescribed value functions passed to the problem.

In [ ]:
# Surface location functions
left = lambda point: jnp.isclose(point[0], 0.0)
y_sides = lambda point: jnp.isclose(point[1], 0.0) | jnp.isclose(point[1], Ly)
z_sides = lambda point: jnp.isclose(point[2], 0.0) | jnp.isclose(point[2], Lz)
zero = lambda point: 0.0

# JAX-FEM boundary condition helper
dirichlet_bc_info = [
    [left, y_sides, z_sides],
    [0, 1, 2],
    [zero, zero, zero],
]

### Define problem

We define a subclass of `jax_fem.problem.Problem` and implement `get_tensor_map` to provide the linear elastic stress. The Poisson's ratio is fixed, while the layerwise Young's modulus is supplied at the element quadrature points. 

In [ ]:
class LinearElasticity(Problem):
    """
    Small-strain isotropic elasticity with a layerwise Young's modulus.
    """

    # Initialize the fixed Poisson's ratio and quadrature-point modulus
    def custom_init(self, nu):
        self.nu = nu
        self.internal_vars = [jnp.ones((self.num_cells, self.fes[0].num_quads))]

    # Define the stress map used by JAX-FEM
    def get_tensor_map(self):
        def stress(u_grad, E):
            strain = 0.5 * (u_grad + u_grad.T)
            mu = E / (2.0 * (1.0 + self.nu))
            lmbda = E * self.nu / ((1.0 + self.nu) * (1.0 - 2.0 * self.nu))
            return lmbda * jnp.trace(strain) * jnp.eye(3) + 2.0 * mu * strain

        return stress

# Problem instance
problem = LinearElasticity(
    mesh,
    vec=3,
    dim=3,
    ele_type=ele_type,
    dirichlet_bc_info=dirichlet_bc_info,
    additional_info=(nu,),
)

### Lumped mass and initial state

The arrays `fe.shape_vals`, `fe.JxW`, and `fe.cells` provide the shape-function values, weighted Jacobians, and element connectivity used to assemble the row-sum lumped nodal mass $m_a=\int_\Omega\rho N_a\,\mathrm d\Omega$. The first fixed-free longitudinal mode defines the initial displacement.

In [ ]:
# Assemble the row-sum lumped nodal masses
fe = problem.fes[0]
element_mass = rho * jnp.einsum("qi,eq->ei", fe.shape_vals, fe.JxW)
nodal_mass = jnp.zeros(fe.num_total_nodes)
nodal_mass = nodal_mass.at[fe.cells.reshape(-1)].add(element_mass.reshape(-1))
mass = jnp.repeat(nodal_mass[:, None], 3, axis=1)

# Build a mask for the constrained displacement components
fixed = jnp.zeros((fe.num_total_nodes, 3), dtype=bool)
for node_inds, vec_inds in zip(fe.node_inds_list, fe.vec_inds_list):
    fixed = fixed.at[node_inds, vec_inds].set(True)

# First fixed-free longitudinal mode
amplitude = 1.0e-3
wave_number = np.pi / (2.0 * Lx)
mode = jnp.sin(wave_number * fe.points[:, 0])
u0 = jnp.zeros((fe.num_total_nodes, 3)).at[:, 0].set(amplitude * mode)

# Center node of the traction-free end
free_end_node = np.argmin(
    np.linalg.norm(points - np.array([Lx, Ly / 2.0, Lz / 2.0]), axis=1)
)

print(f"Degrees of freedom: {3 * fe.num_total_nodes}")

### Explicit time integration

A fixed time grid spanning one period of the first mode is selected according to the CFL stability condition. At each time step, `problem.compute_residual_vars` assembles the internal force from the current displacement and the quadrature-point modulus. The functions `acceleration` and `step` implement the common operations, and `jax.lax.scan` advances the uniform-material problem while retaining its displacement history.

In [ ]:
# Construct a fixed time grid for one period of the first mode
cfl = 0.4
wave_speed = np.sqrt(E0 * (1.0 - nu) / ((1.0 + nu) * (1.0 - 2.0 * nu) * rho))
period = 2.0 * np.pi / (wave_speed * wave_number)
dt_limit = cfl * min(Lx / Nx, Ly / Ny, Lz / Nz) / wave_speed
num_steps = int(np.ceil(period / dt_limit))
dt = period / num_steps
times = np.arange(num_steps + 1) * dt

print(f"Time steps: {num_steps}")
print(f"Time-step size: {dt:.6e}")

v0 = jnp.zeros_like(u0)

# Compute the unconstrained nodal acceleration
def acceleration(u, E_quad):
    internal_force = problem.compute_residual_vars(
        [u], [E_quad], problem.internal_vars_surfaces
    )[0]
    return jnp.where(fixed, 0.0, -internal_force / mass)

# Advance one central-difference step
def step(carry, E_quad, output_nodes=None):
    u_previous, u = carry
    u_next = 2.0 * u - u_previous + dt**2 * acceleration(u, E_quad)
    u_next = jnp.where(fixed, 0.0, u_next)
    output = u_next if output_nodes is None else u_next[output_nodes, 0]
    return (u, u_next), output

# Integrate the uniform-material problem
E_quad = E0 * jnp.ones((problem.num_cells, fe.num_quads))
u_previous = u0 - dt * v0 + 0.5 * dt**2 * acceleration(u0, E_quad)
u_previous = jnp.where(fixed, 0.0, u_previous)
_, displacement_history = jax.lax.scan(
    lambda carry, _: step(carry, E_quad),
    (u_previous, u0), None, length=num_steps,
)
displacement_history = jnp.concatenate((u0[None, ...], displacement_history))
numerical_signal = displacement_history[:, free_end_node, 0]

### Analytical verification

For the uniform modulus $E_0$, the exact free-end displacement is $u_x(L_x,t)=A\cos(c_p k t)$. This analytical solution is used to verify the numerical response. The numerical wave speed is obtained from the first zero crossing of the mass-weighted modal displacement. The maximum displacement error measures the solution accuracy, while the kinetic, strain, and total energies verify the expected energy exchange and conservation.

In [ ]:
# Analytical free-end response
analytical_signal = amplitude * np.cos(wave_speed * wave_number * times)
numerical_signal = np.asarray(numerical_signal)
normalized_error = np.max(np.abs(numerical_signal - analytical_signal)) / amplitude

# Estimate the wave speed from the mass-weighted modal displacement
modal_signal = np.sum(
    np.asarray(nodal_mass)[None, :] * np.asarray(displacement_history[:, :, 0])
    * np.asarray(mode)[None, :], axis=1
)
crossing = np.flatnonzero(modal_signal[1:] <= 0.0)[0]
fraction = modal_signal[crossing] / (modal_signal[crossing] - modal_signal[crossing + 1])
zero_time = times[crossing] + fraction * dt
measured_speed = (np.pi / (2.0 * zero_time)) / wave_number

# Kinetic, strain, and total energies
velocity = jnp.zeros_like(displacement_history)
velocity = velocity.at[0].set(v0)
velocity = velocity.at[1:-1].set(
    (displacement_history[2:] - displacement_history[:-2]) / (2.0 * dt)
)
velocity = velocity.at[-1].set(
    (3.0 * displacement_history[-1] - 4.0 * displacement_history[-2]
     + displacement_history[-3]) / (2.0 * dt)
)
kinetic_energy = 0.5 * jnp.sum(mass[None, ...] * velocity**2, axis=(1, 2))

u_grad = jnp.einsum("teni,eqnj->teqij", displacement_history[:, fe.cells], fe.shape_grads)
strain = 0.5 * (u_grad + jnp.swapaxes(u_grad, -1, -2))
mu = E0 / (2.0 * (1.0 + nu))
lmbda = E0 * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
strain_energy_density = (
    0.5 * lmbda * jnp.trace(strain, axis1=-2, axis2=-1)**2
    + mu * jnp.sum(strain**2, axis=(-1, -2))
)
strain_energy = jnp.sum(strain_energy_density * fe.JxW[None, ...], axis=(1, 2))
total_energy = kinetic_energy + strain_energy
energy_drift = jnp.max(jnp.abs(total_energy / total_energy[0] - 1.0))

# Report the forward-solution errors
print(f"Analytical wave speed: {wave_speed:.8f}")
print(f"Measured wave speed:   {measured_speed:.8f}")
print(f"Relative speed error:  {abs(measured_speed / wave_speed - 1.0):.3e}")
print(f"Maximum normalized displacement error: {normalized_error:.3e}")
print(f"Maximum relative total-energy drift: {float(energy_drift):.3e}")

In [ ]:
# Compare the free-end responses and energy histories
plot_style = {"text.usetex": True, "font.family": "serif", "font.size": 18,
              "axes.labelsize": 18, "xtick.labelsize": 16,
              "ytick.labelsize": 16, "legend.fontsize": 15}
blue, red = "#1f77b4", "#d62728"
with plt.rc_context(plot_style):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(times, numerical_signal, color=blue, label="JAX-FEM", lw=2.4)
    axes[0].plot(times, analytical_signal, "--", color=red,
                 label="Analytical", lw=2.4)
    axes[0].set(xlabel="Time", ylabel=r"Free-end displacement $u_x$")
    axes[0].legend()

    axes[1].plot(times, kinetic_energy, color=blue, label="Kinetic", lw=2.4)
    axes[1].plot(times, strain_energy, color=red, label="Strain", lw=2.4)
    axes[1].plot(times, total_energy, "k--", label="Total", lw=2.2)
    axes[1].set(xlabel="Time", ylabel="Energy")
    axes[1].legend()
    for ax in axes:
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0), useMathText=True)
        ax.grid(alpha=0.25)
    fig.tight_layout()
    root = os.getcwd() if os.path.isdir("input") else os.path.join(os.getcwd(), "codes")
    output_dir = os.path.join(root, "output")
    os.makedirs(output_dir, exist_ok=True)
    np.savez(
        os.path.join(output_dir, "exp_forward.npz"),
        times=times, numerical_signal=numerical_signal,
        analytical_signal=analytical_signal, modal_signal=modal_signal,
        kinetic_energy=np.asarray(kinetic_energy),
        strain_energy=np.asarray(strain_energy), total_energy=np.asarray(total_energy),
        analytical_wave_speed=wave_speed, numerical_wave_speed=measured_speed,
        normalized_error=normalized_error, energy_drift=float(energy_drift),
    )
    fig.savefig(os.path.join(output_dir, "exp_forward.png"),
                dpi=600, bbox_inches="tight")
    plt.show()

### Synthetic observations

The target Young's modulus $E^*(x)=E_0(1-0.4x/L_x)$ is represented by one constant value in each axial element layer. Solving the forward problem with $E^*$ gives the noise-free axial displacement observations at the centerline probes.

In [ ]:
# Assign one modulus parameter to each axial element layer
cell_centers = jnp.mean(fe.points[fe.cells], axis=1)
cell_layer = jnp.clip(jnp.floor(cell_centers[:, 0] * Nx / Lx).astype(int), 0, Nx - 1)
x_layers = (jnp.arange(Nx) + 0.5) * Lx / Nx

# Locate one centerline observation node on each free cross-section
x_planes = np.linspace(0.0, Lx, Nx + 1)
probe_nodes = np.array([
    np.argmin(np.linalg.norm(points - [x, Ly / 2.0, Lz / 2.0], axis=1))
    for x in x_planes[1:]
])

# Assemble the forward map required by the inverse problem
def simulate(E, steps=num_steps):
    E_quad = jnp.repeat(E[cell_layer, None], fe.num_quads, axis=1)
    u_previous = u0 - dt * v0 + 0.5 * dt**2 * acceleration(u0, E_quad)
    u_previous = jnp.where(fixed, 0.0, u_previous)
    _, signal = jax.lax.scan(
        lambda carry, _: step(carry, E_quad, probe_nodes),
        (u_previous, u0), None, length=steps,
    )
    return jnp.concatenate((u0[probe_nodes, 0][None, :], signal), axis=0)

# Linear target field
E_target = E0 * (1.0 - 0.4 * x_layers / Lx)

# Generate synthetic observations from the target modulus field
observed_signal = simulate(E_target)
print(f"Material layers: {Nx}")
print(f"Observation points: {probe_nodes.size}")

### Identification

Starting from the uniform field $E=E_0$, MoMo-Adam identifies the $N_x$ layer values by minimizing the normalized displacement mismatch. Reverse-mode AD differentiates the loss through the explicit recurrence. The time steps are grouped into blocks, and `jax.checkpoint` is applied to each block to reduce the intermediate values retained for backpropagation.

In [ ]:
# Checkpointed forward map used during backpropagation
checkpoint_interval = 80
def simulate_checkpointed(E, interval=checkpoint_interval, steps=num_steps):
    # Divide the complete trajectory into equal checkpoint blocks
    if steps % interval != 0:
        raise ValueError("steps must be divisible by interval")

    # Expand the layer values to all element quadrature points
    E_quad = jnp.repeat(E[cell_layer, None], fe.num_quads, axis=1)

    # Initialize the fictitious previous displacement
    u_previous = u0 - dt * v0 + 0.5 * dt**2 * acceleration(u0, E_quad)
    u_previous = jnp.where(fixed, 0.0, u_previous)

    # Advance one block and retain the probe displacements
    def advance_block(carry, _):
        return jax.lax.scan(
            lambda state, _: step(state, E_quad, probe_nodes),
            carry, None, length=interval,
        )

    # Advance the prescribed number of checkpoint blocks
    num_blocks = steps // interval
    _, signal = jax.lax.scan(
        jax.checkpoint(advance_block), (u_previous, u0), None,
        length=num_blocks,
    )

    # Flatten the block dimension and prepend the initial observations
    signal = signal.reshape(steps, probe_nodes.size)
    return jnp.concatenate((u0[probe_nodes, 0][None, :], signal), axis=0)

# Normalized displacement loss and its reverse-mode gradient
scale = jnp.sqrt(jnp.mean(observed_signal**2)) + 1.0e-8
def loss(E):
    prediction = simulate_checkpointed(E)
    return jnp.mean(((prediction - observed_signal) / scale)**2)

value_and_grad = jax.jit(jax.value_and_grad(loss))

In [ ]:
# Adaptive optimizer with the known lower loss bound
optimizer = optax.contrib.momo_adam(
    learning_rate=1.0, lower_bound=0.0, weight_decay=0.0
)

# Uniform initial field and optimizer state
E_initial = E0 * jnp.ones(Nx)
E = E_initial
optimizer_state = optimizer.init(E)
loss_history = []
E_error_history = []

# Identify the layerwise modulus field
for iteration in range(200):
    value, gradient = value_and_grad(E)
    updates, optimizer_state = optimizer.update(
        gradient, optimizer_state, E, value=value
    )
    E = jnp.clip(optax.apply_updates(E, updates), 0.5 * E0, E0)
    loss_history.append(float(value))
    E_error = jnp.linalg.norm(E - E_target) / jnp.linalg.norm(E_target)
    E_error_history.append(float(E_error))
    if iteration % 20 == 0:
        print(f"Step {iteration:02d}: loss = {float(value):.3e}, E error = {float(E_error):.3e}")

# Evaluate the initial and identified displacement histories
E_identified = E
identified_signal = simulate(E_identified)
identified_signal.block_until_ready()
initial_signal = simulate(E_initial)
initial_signal.block_until_ready()
print(f"Final relative E error: {E_error_history[-1]:.3e}")

In [ ]:
# Compare the target field, displacement histories, and loss convergence
np.savez(
    os.path.join(output_dir, "exp_identification.npz"),
    x_layers=np.asarray(x_layers), E_target=np.asarray(E_target),
    E_initial=np.asarray(E_initial), E_identified=np.asarray(E_identified),
    times=times, observed_signal=np.asarray(observed_signal),
    initial_signal=np.asarray(initial_signal),
    identified_signal=np.asarray(identified_signal),
    loss_history=np.asarray(loss_history),
    E_error_history=np.asarray(E_error_history), probe_nodes=probe_nodes,
)

with plt.rc_context(plot_style):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(x_layers, E_target, label="Target", lw=2.2)
    axes[0].plot(x_layers, E_identified, "--", label="Identified", lw=2)
    axes[0].set(xlabel=r"Axial coordinate $x$", ylabel=r"Young's modulus $E$")
    axes[0].legend()

    axes[1].plot(times, observed_signal[:, -1], label="Observed", lw=2.2)
    axes[1].plot(times, initial_signal[:, -1], "--", label="Initial", lw=1.8)
    axes[1].plot(times, identified_signal[:, -1], "-.", label="Identified", lw=1.8)
    axes[1].set(xlabel="Time", ylabel=r"Free-end displacement $u_x$")
    axes[1].legend()

    axes[2].semilogy(np.arange(len(loss_history)), loss_history, lw=2.2)
    axes[2].set(xlabel="Optimization step", ylabel=r"Loss $\mathcal{L}$")

    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "exp_identification.png"),
                dpi=600, bbox_inches="tight")
    plt.show()

### Checkpoint performance

We fix the observation horizon at sixteen periods and vary the checkpoint interval over a geometric sequence. This directly exposes the memory--runtime trade-off controlled by checkpoint block length. Each configuration runs in a fresh process to reduce cross-configuration allocator effects. After compilation, one loss-and-gradient evaluation is sampled for peak process RSS. Steady-state runtime is measured separately from three additional synchronized evaluations and reported as their median.

In [ ]:
import json
import subprocess
import sys

# Fix a long trajectory and vary the checkpoint block length
benchmark_steps = 16 * num_steps
intervals = [0, 6400, 3200, 1600, 800]
labels = ["None", "6400", "3200", "1600", "800"]
peak_rss = np.zeros(len(intervals))
execution_increment = np.zeros_like(peak_rss)
runtime = np.zeros_like(peak_rss)
runtime_samples = np.zeros((len(intervals), 3))

# Locate the notebook and the isolated-process benchmark helper
notebook_path = os.path.join(root, "tutorial_explicit.ipynb")
benchmark_script = os.path.join(root, "benchmark_explicit_memory.py")

# Measure each configuration in a fresh process
for index, interval in enumerate(intervals):
    completed = subprocess.run(
        [sys.executable, benchmark_script, "--notebook", notebook_path,
         "--steps", str(benchmark_steps), "--interval", str(interval),
         "--repeats", "3"],
        check=True, capture_output=True, text=True,
    )
    result = json.loads(completed.stdout.strip().splitlines()[-1])
    peak_rss[index] = result["peak_rss_bytes"] / 2**30
    execution_increment[index] = result["peak_increment_bytes"] / 2**30
    runtime[index] = result["runtime_seconds"]
    runtime_samples[index] = result["runtime_samples_seconds"]

# Report the absolute peak process RSS used in the comparison plot
for label, memory, elapsed in zip(labels, peak_rss, runtime):
    print(f"Interval {label}: {memory:.2f} GiB, {elapsed:.2f} s")

In [ ]:
# Plot the memory and runtime trade-off
np.savez(
    os.path.join(output_dir, "exp_checkpoint.npz"),
    benchmark_steps=benchmark_steps,
    checkpoint_intervals=np.asarray(intervals),
    labels=np.asarray(labels), peak_rss=peak_rss,
    execution_increment=execution_increment, runtime=runtime,
    runtime_samples=runtime_samples,
)

checkpoint_plot_style = {
    **plot_style,
    "font.size": 19,
    "axes.labelsize": 21,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 15,
}
with plt.rc_context(checkpoint_plot_style):
    fig, (memory_ax, runtime_ax) = plt.subplots(1, 2, figsize=(10.8, 5.0))
    baseline_rss = peak_rss - execution_increment

    # Decompose the absolute peak RSS into the post-compilation baseline
    # and the additional working memory used by loss-and-gradient execution.
    x = np.arange(len(labels))
    bar_width = 0.40
    memory_ax.bar(
        x, baseline_rss, width=bar_width, color="#E8E5E0", edgecolor="white",
        linewidth=0.5, label="Post-compilation baseline",
    )
    working_bars = memory_ax.bar(
        x, execution_increment, width=bar_width, bottom=baseline_rss,
        color="#496A81",
        edgecolor="white", linewidth=0.5,
        label="Execution working memory",
    )
    runtime_range = np.vstack((
        runtime - runtime_samples.min(axis=1),
        runtime_samples.max(axis=1) - runtime,
    ))
    runtime_bars = runtime_ax.bar(
        x, runtime, width=bar_width, color="#A65A64",
        edgecolor="white", linewidth=0.5, yerr=runtime_range,
        capsize=4, error_kw={"elinewidth": 1.1, "capthick": 1.1},
    )

    memory_ax.set(
        xlabel="Checkpoint interval (time steps)",
        ylabel="Peak process RSS (GiB)",
        xticks=x, xticklabels=labels,
        ylim=(0.0, 1.08 * peak_rss.max()),
    )
    runtime_ax.set(
        xlabel="Checkpoint interval (time steps)",
        ylabel="Median loss-and-gradient time (s)",
        xticks=x, xticklabels=labels,
    )
    memory_ax.bar_label(
        working_bars, labels=[f"{value:.2f}" for value in peak_rss],
        padding=3, fontsize=16,
    )
    runtime_ax.bar_label(
        runtime_bars, labels=[f"{value:.1f}" for value in runtime],
        padding=5, fontsize=16,
    )
    memory_ax.text(-0.10, 1.22, "(a)", transform=memory_ax.transAxes,
                   ha="left", va="bottom", fontsize=19, clip_on=False)
    runtime_ax.text(-0.10, 1.22, "(b)", transform=runtime_ax.transAxes,
                    ha="left", va="bottom", fontsize=19, clip_on=False)
    memory_ax.legend(loc="lower center", bbox_to_anchor=(0.52, 1.02),
                     frameon=False)
    for ax in (memory_ax, runtime_ax):
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
    fig.subplots_adjust(left=0.11, right=0.985, bottom=0.18,
                        top=0.76, wspace=0.34)
    checkpoint_output_dir = "/Users/bytedance/Desktop"
    os.makedirs(checkpoint_output_dir, exist_ok=True)
    fig.savefig(os.path.join(checkpoint_output_dir, "exp_checkpoint.pdf"),
                dpi=600, bbox_inches="tight")
    plt.show()